# Face Mask Detection using CNN


## Problem Statement
Build a binary image classifier to detect whether a person
is wearing a face mask or not, using a custom CNN architecture.

## Dataset
- Source: Kaggle — Face Mask 12K Images Dataset
- Classes: WithMask / WithoutMask
- Train: ~10,000 images | Validation: ~800 images


## Approach
- Built custom CNN from scratch using TensorFlow/Keras
- Architecture: 3 Conv blocks (BatchNorm + MaxPool) → Dense layers
- Used EarlyStopping to prevent overfitting
- Data pipeline optimized with cache() and prefetch()

In [1]:
# Dataset setup:
# Download manually from:
# https://www.kaggle.com/datasets/ashishjangra27/face-mask-12k-images-dataset
# Extract it in your project folder before running this notebook

mv: cannot stat 'kaggle.json': No such file or directory
chmod: cannot access '/root/.kaggle/kaggle.json': No such file or directory


In [4]:
import tensorflow as tf
from tensorflow import keras
from keras import Sequential
from keras.layers import Dense,Conv2D,BatchNormalization,MaxPool2D,Flatten,Dropout
from keras.callbacks import EarlyStopping

In [5]:
train_dir = "Face Mask Dataset/Train"
val_dir = "Face Mask Dataset/Validation"

train_ds = keras.utils.image_dataset_from_directory(
    directory=train_dir,
    labels="inferred",
    label_mode="int",
    image_size=(256, 256),
    batch_size=32
)

validation_ds = keras.utils.image_dataset_from_directory(
    directory=val_dir,
    labels="inferred",
    label_mode="int",
    image_size=(256, 256),
    batch_size=32
)

Found 10000 files belonging to 2 classes.
Found 800 files belonging to 2 classes.


In [6]:
def process(image, label):
    image = tf.cast(image, tf.float32) / 255.0
    return image, label

In [ ]:
train_ds = train_ds.map(process).cache().prefetch(buffer_size=tf.data.AUTOTUNE)
validation_ds = validation_ds.map(process).prefetch(buffer_size=tf.data.AUTOTUNE)

## Model Architecture
3 convolutional blocks with increasing filters (32 → 64 → 128),
each followed by BatchNormalization and MaxPooling.
Fully connected layers with Dropout(0.3) for regularization.
Binary output with sigmoid activation.

In [12]:
model=Sequential()

model.add(Conv2D(32,kernel_size=(3,3),padding="valid",activation="relu",input_shape=(256,256,3)))
model.add(BatchNormalization())
model.add(MaxPool2D(pool_size=(2,2),strides=2,padding="valid"))



model.add(Conv2D(64,kernel_size=(3,3),padding="valid",activation="relu"))
model.add(BatchNormalization())
model.add(MaxPool2D(pool_size=(2,2),strides=2,padding="valid"))


model.add(Conv2D(128,kernel_size=(3,3),padding="valid",activation="relu",))
model.add(BatchNormalization())
model.add(MaxPool2D(pool_size=(2,2),strides=2,padding="valid"))

model.add(Flatten())

model.add(Dense(128,activation="relu"))
model.add(Dropout(0.3))
model.add(Dense(64,activation="relu"))
model.add(Dropout(0.3))
model.add(Dense(1,activation="sigmoid"))



In [8]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 254, 254, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 254, 254, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 127, 127, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 125, 125, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 125, 125, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 62, 62, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 60, 60, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 60, 60, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 30, 30, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 115200)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │    14,745,728 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 14,848,193 (56.64 MB)

 Trainable params: 14,847,745 (56.64 MB)

 Non-trainable params: 448 (1.75 KB)

In [13]:
callback = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

## Training
- Optimizer: Adam
- Loss: Binary Crossentropy
- EarlyStopping on val_loss with patience=3
- Max epochs: 25

In [14]:
model.compile(loss="binary_crossentropy",optimizer="adam",metrics=["accuracy"])

In [ ]:
history=model.fit(train_ds,epochs=25,validation_data=validation_ds,callbacks=callback)

Epoch 1/25
 90/313 ━━━━━━━━━━━━━━━━━━━━ 24:17 7s/step - accuracy: 0.8952 - loss: 2.0688

In [ ]:
import matplotlib.pyplot as plt
plt.plot(history.history["loss"])
plt.plot(history.history["val_loss"])
plt.title("Loss_curve")
plt.show()

In [ ]:
plt.plot(history.history["accuracy"])
plt.plot(history.history["val_accuracy"])
plt.title("Accuracy_curve")
plt.show()

In [ ]:
loss,acc=model.evaluate(validation_ds)
print(f"Validation Accuracy: {acc:.4f}")

In [ ]:
# Predict on a sample image
import numpy as np
from keras.preprocessing import image

def predict_mask(img_path):
    img = image.load_img(img_path, target_size=(256, 256))
    img_array = image.img_to_array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)
    prediction = model.predict(img_array)[0][0]
    label = "No Mask" if prediction > 0.5 else "Mask"
    confidence = prediction if prediction > 0.5 else 1 - prediction
    print(f"Prediction: {label} ({confidence*100:.2f}% confidence)")

In [ ]:
import os
os.makedirs("models", exist_ok=True)

model.save("models/face_mask_model.h5")